# Praktikum Data Mining
## Evaluasi Model Klasifikasi: Confusion Matrix, Accuracy, Precision, Recall, dan F1-Score

**Nama:** Fauziano Silva Dwinanta  
**NIM:** 24343115  
**Program Studi:** S1 Teknik Informatika  
**Mata Kuliah:** Data Mining  

Notebook ini mengikuti Modul Praktikum Data Mining UNP 2026. Dataset yang digunakan merupakan data sintetis transaksi pembayaran digital; `isFraud = 1` berarti fraud dan `isFraud = 0` berarti normal.

**Alur:** membangkitkan dataset → membagi data → melatih Decision Tree → menghitung confusion matrix dan metrik → visualisasi → eksperimen baseline dan threshold.

## 1. Persiapan Library
Jalankan sel berikut. Jika ada library yang belum tersedia, aktifkan sel instalasi di bawah.

In [ ]:
# Jika diperlukan, hapus tanda komentar lalu jalankan:
# %pip install numpy pandas scikit-learn matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

print("Library berhasil diimpor.")

## 2. Membangkitkan Dataset Sintetis (Bagian 4.1)
Dataset dibuat secara deterministik menggunakan `random_state`/seed 42 agar hasil dapat direproduksi.

In [ ]:
rng = np.random.default_rng(42)
n = 500

amount = rng.lognormal(mean=5.0, sigma=1.0, size=n)
hour = rng.integers(0, 24, n)
new_device = rng.binomial(1, 0.25, n)
foreign_ip = rng.binomial(1, 0.12, n)

score = -4.5 + 0.45*(amount > 300) + 1.1*new_device + 1.5*foreign_ip + 0.7*(hour < 5)
prob = 1/(1+np.exp(-score))
isFraud = rng.binomial(1, prob)

df = pd.DataFrame({
    'amount': amount.round(2),
    'hour': hour,
    'new_device': new_device,
    'foreign_ip': foreign_ip,
    'isFraud': isFraud
})

print("Lima baris pertama dataset:")
display(df.head())
print("\nDistribusi kelas:")
print(df['isFraud'].value_counts().sort_index().rename(index={0:'Normal (0)', 1:'Fraud (1)'}))
print("\nProporsi kelas:")
display(df['isFraud'].value_counts(normalize=True).sort_index().rename(index={0:'Normal (0)', 1:'Fraud (1)'}).to_frame('Proporsi'))

## 3. Membagi Data dan Melatih Model (Bagian 4.2)
Data dibagi menjadi 70% data latih dan 30% data uji. `stratify=y` menjaga proporsi kelas, sedangkan `class_weight='balanced'` memberi bobot lebih besar pada kelas minoritas.

In [ ]:
X = df.drop(columns='isFraud')
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model = DecisionTreeClassifier(
    max_depth=4,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Jumlah data latih:", len(y_train))
print("Jumlah data uji:", len(y_test))
print("Data uji:", len(y_test))

## 4. Confusion Matrix (Bagian 4.3)
Dengan `labels=[0,1]`, susunan matriks scikit-learn adalah `[[TN, FP], [FN, TP]]`.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix:")
print(cm)
print(f"TP = {tp}")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")

## 5. Perhitungan Manual dan Verifikasi scikit-learn (Bagian 4.4)
Rumus dihitung dari nilai TP, TN, FP, dan FN. Pemeriksaan penyebut nol dilakukan agar program tidak mengalami pembagian dengan nol.

In [ ]:
total = tp + tn + fp + fn

accuracy = (tp + tn) / total if total else 0
precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print("Hasil perhitungan manual:")
print(f"Accuracy  = {accuracy:.3f}")
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1-score  = {f1:.3f}")

sk_accuracy = accuracy_score(y_test, y_pred)
sk_precision = precision_score(y_test, y_pred, zero_division=0)
sk_recall = recall_score(y_test, y_pred, zero_division=0)
sk_f1 = f1_score(y_test, y_pred, zero_division=0)

print("\nHasil scikit-learn:")
print(f"Accuracy  = {sk_accuracy:.3f}")
print(f"Precision = {sk_precision:.3f}")
print(f"Recall    = {sk_recall:.3f}")
print(f"F1-score  = {sk_f1:.3f}")

print("\nVerifikasi:")
print("Semua hasil manual sama dengan scikit-learn:",
      np.allclose([accuracy, precision, recall, f1],
                  [sk_accuracy, sk_precision, sk_recall, sk_f1]))

## 6. Visualisasi Confusion Matrix (Bagian 4.5)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Normal', 'Fraud']
)
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix – Deteksi Fraud')
plt.tight_layout()
plt.show()

print(f"FN = {fn}: transaksi fraud diprediksi sebagai normal sehingga berpotensi lolos dan menimbulkan kerugian.")
print(f"FP = {fp}: transaksi normal diprediksi sebagai fraud sehingga pelanggan dapat terganggu atau transaksi tertahan.")

## 7. Interpretasi Hasil (Bagian 4.6)
- **TP:** transaksi fraud berhasil dikenali sebagai fraud.
- **TN:** transaksi normal berhasil dikenali sebagai normal.
- **FP:** transaksi normal salah ditandai sebagai fraud (alarm palsu).
- **FN:** transaksi fraud salah diprediksi sebagai normal (fraud luput).

Dalam deteksi fraud, FN dapat menyebabkan kerugian karena transaksi berbahaya lolos. FP dapat mengganggu pelanggan karena transaksi normal tertahan atau diperiksa. Recall yang tinggi membantu menemukan lebih banyak fraud, tetapi dapat meningkatkan alarm palsu. Karena itu, pemilihan metrik perlu mempertimbangkan biaya masing-masing kesalahan.

## 8. Eksperimen Baseline Semua Normal (Bagian 4.7)
Baseline selalu memprediksi seluruh transaksi sebagai normal. Bandingkan akurasinya dengan kemampuan mendeteksi kelas fraud pada Decision Tree.

In [ ]:
baseline = np.zeros(len(y_test), dtype=int)

print("Baseline accuracy:", accuracy_score(y_test, baseline))
print("\nClassification report – Baseline semua normal:")
print(classification_report(y_test, baseline, labels=[0, 1],
                            target_names=['Normal', 'Fraud'], zero_division=0))

print("Classification report – Decision Tree:")
print(classification_report(y_test, y_pred, labels=[0, 1],
                            target_names=['Normal', 'Fraud'], zero_division=0))

## 9. Eksperimen Pengaruh Ambang Prediksi (Bagian 4.8)
Probabilitas kelas fraud dibandingkan dengan empat threshold. Threshold lebih rendah cenderung membuat model lebih mudah memberi label fraud, tetapi hasil precision dan recall tetap harus dilihat dari output aktual.

In [ ]:
y_prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]

threshold_results = []
for threshold in [0.2, 0.4, 0.5, 0.7]:
    pred = (y_prob >= threshold).astype(int)
    p = precision_score(y_test, pred, zero_division=0)
    r = recall_score(y_test, pred, zero_division=0)
    threshold_results.append({
        'Threshold': threshold,
        'Precision': p,
        'Recall': r
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.style.format({'Precision':'{:.3f}', 'Recall':'{:.3f}'}))

## 10. Tugas Praktikum
### Tugas 1 – Hitung Manual
Diketahui TP=24, TN=156, FP=12, FN=8.

In [ ]:
tp_t1, tn_t1, fp_t1, fn_t1 = 24, 156, 12, 8
total_t1 = tp_t1 + tn_t1 + fp_t1 + fn_t1

cm_t1 = np.array([[tn_t1, fp_t1],
                  [fn_t1, tp_t1]])

acc_t1 = (tp_t1 + tn_t1) / total_t1
prec_t1 = tp_t1 / (tp_t1 + fp_t1)
rec_t1 = tp_t1 / (tp_t1 + fn_t1)
f1_t1 = 2 * prec_t1 * rec_t1 / (prec_t1 + rec_t1)

print("Confusion Matrix [ [TN, FP], [FN, TP] ]:")
print(cm_t1)
print(f"Accuracy  = ({tp_t1}+{tn_t1})/{total_t1} = {acc_t1:.4f} ({acc_t1*100:.2f}%)")
print(f"Precision = {tp_t1}/({tp_t1}+{fp_t1}) = {prec_t1:.4f} ({prec_t1*100:.2f}%)")
print(f"Recall    = {tp_t1}/({tp_t1}+{fn_t1}) = {rec_t1:.4f} ({rec_t1*100:.2f}%)")
print(f"F1-score  = {f1_t1:.4f} ({f1_t1*100:.2f}%)")

**Interpretasi Tugas 1:**
- **FP = 12:** terdapat 12 transaksi normal yang keliru ditandai sebagai fraud. Dampaknya, transaksi pelanggan yang sah dapat tertahan atau perlu verifikasi.
- **FN = 8:** terdapat 8 transaksi fraud yang tidak terdeteksi dan dianggap normal. Dampaknya, transaksi berpotensi menimbulkan kerugian.

### Tugas 2 – Verifikasi Python
Bagian 2 sampai 6 telah menjalankan dataset, confusion matrix, perhitungan manual, dan verifikasi scikit-learn. Nilai TP/TN/FP/FN serta metrik diambil langsung dari hasil eksekusi, bukan ditebak.

In [ ]:
verification = pd.DataFrame({
    'Metrik': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Perhitungan Manual': [accuracy, precision, recall, f1],
    'scikit-learn': [sk_accuracy, sk_precision, sk_recall, sk_f1]
})
display(verification.style.format({
    'Perhitungan Manual':'{:.4f}',
    'scikit-learn':'{:.4f}'
}))
print("Kesimpulan: hasil perhitungan manual dan scikit-learn sesuai apabila nilai pada kedua kolom sama.")

### Tugas 3 – Evaluasi Data Tidak Seimbang

Baseline semua normal dapat memperoleh accuracy yang terlihat tinggi apabila jumlah data normal jauh lebih banyak daripada fraud. Namun, baseline tidak pernah mendeteksi fraud sehingga **recall kelas fraud = 0**. Oleh karena itu, accuracy tidak cukup digunakan sendirian pada data tidak seimbang. Precision, recall, F1-score, dan confusion matrix perlu diperiksa, terutama untuk kelas fraud.

### Tugas 4 – Eksperimen dan Rekomendasi
Tabel hasil precision dan recall empat threshold ditampilkan pada Bagian 9. Untuk skenario biaya FN lebih tinggi daripada FP, threshold dipilih dengan mempertimbangkan recall fraud yang tinggi (mengurangi fraud yang lolos), sambil tetap memperhatikan penurunan precision dan bertambahnya alarm palsu. Gunakan nilai pada tabel hasil eksekusi untuk menentukan threshold yang paling sesuai; jangan mengasumsikan bahwa perubahan threshold selalu menghasilkan pola monoton pada dataset kecil.

In [ ]:
# Rekomendasi berbasis hasil: urutkan threshold berdasarkan recall tertinggi,
# lalu precision sebagai pertimbangan berikutnya.
recommendation = threshold_df.sort_values(
    by=['Recall', 'Precision'],
    ascending=[False, False]
).iloc[0]

print("Threshold dengan recall tertinggi (tie-break precision):")
print(f"Threshold = {recommendation['Threshold']}")
print(f"Precision = {recommendation['Precision']:.3f}")
print(f"Recall    = {recommendation['Recall']:.3f}")
print("\nCatatan: keputusan akhir tetap mempertimbangkan biaya alarm palsu (FP) dan fraud yang lolos (FN).")

## 11. Kesimpulan

Evaluasi model klasifikasi tidak cukup hanya melihat accuracy. Confusion matrix menunjukkan jenis kesalahan model, sedangkan precision mengukur ketepatan prediksi fraud, recall mengukur kemampuan menemukan fraud aktual, dan F1-score merangkum precision serta recall. Pada kasus deteksi fraud, recall penting karena FN dapat menimbulkan kerugian, tetapi precision juga perlu dijaga agar transaksi normal tidak terlalu sering ditandai sebagai fraud.

**Daftar Pustaka**
1. Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3rd ed.). Morgan Kaufmann.
2. Tan, P.-N., Steinbach, M., Karpatne, A., & Kumar, V. (2019). *Introduction to Data Mining* (2nd ed.). Pearson.
3. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
4. scikit-learn developers. *Model evaluation: quantifying the quality of predictions*. https://scikit-learn.org/stable/modules/model_evaluation.html